In [1]:
import numpy as np
import pdb
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..')) 
sys.path.append(project_root)  # Add folder_a to sys.path
sys.path.append(os.path.join(project_root, '../..', 'Networks'))  # Add folder_b to sys.path
import matplotlib.pyplot as plt
import torch
import torch.optim.swa_utils as swa_utils

from torch import nn
import copy

from torch import profiler
from tqdm import tqdm
import numpy as np

import pdb
import torchsde
from Networks.utils import get_gpu_memory,objective,update_ema, adj_lattice, criticalK, order_param,  load_clean_state_dict, sample_kuramoto,loss_compute_kur,save_model
from helpers import  KuramotoHyperUniversal,  find_params
import psutil
from adabelief_pytorch import AdaBelief
import gc
from IPython.display import clear_output

is_cuda = torch.cuda.is_available()
device = 'cuda' if is_cuda else 'cpu'
#device = 'cpu'
if not is_cuda:
    print("Warning: CUDA not available; falling back to CPU but this is likely to be very slow.")
gc.collect()  
torch.cuda.empty_cache()
path=os.path.abspath(os.getcwd())
print(get_gpu_memory())

[{'device': 0, 'total_memory': 24564.0, 'free_memory': 19492.125, 'used_memory': 5071.875}]


In [5]:
state_size=64

batch_size=4*256
weight=10**(-4)
scale=5
scale_kin=5

Astored=adj_lattice(state_size, "erdos", 0.3)
%store -r Astored
Kcrit, L=criticalK(Astored,state_size, scale)
Kcrit = 0.01* Kcrit


A=torch.tensor(Astored).to(torch.float32).to(device)
L=torch.tensor(L).to(torch.float32).to(device)

print(get_gpu_memory())


print(Kcrit)

[{'device': 0, 'total_memory': 24564.0, 'free_memory': 19295.8125, 'used_memory': 5268.1875}]
0.841310456374586


In [15]:
gc.collect()
torch.cuda.empty_cache()            

global_means=[]
global_stds=[]
global_pnumel=[]

dist = torch.distributions.Normal(0, 1)
t_size=32
iter = 100
t_total=1

t = np.linspace(0, iter, iter)

ts_exp = torch.linspace(0, t_total, t_size).to(device)

for layers in range(3,5):

    skip=False
    strategy="tHNC"
   
    SMLt= KuramotoHyperUniversal(batch_size, state_size, Kcrit,A, layers,skip,strategy,device).to(device)
    load_clean_state_dict(SMLt,  path+"/" + strategy + "ema" + str(layers)+ ".pth")
    total_params = sum(p.numel() for p in SML.poly.hhnet.parameters())
    global_pnumel.append(total_params)


    orparam=0
    
    count=0
    mean=0
    M2=0
    for j in tqdm(range(iter)):
        freqs, ys_exp= sample_kuramoto(state_size,batch_size, dist, scale, scale_kin, device)

        SML.freqs=freqs
       
        
        ys_tray = torchsde.sdeint_adjoint(SML, ys_exp, ts_exp,method='reversible_heun', dt=ts_exp[1]-ts_exp[0],
                                     adjoint_method='adjoint_reversible_heun',)
    
        
        obj= objective(ys_tray[-1,:,0:state_size],A)
        dif= weight*ys_tray[-1,:,-1]     
        lossvec=obj+dif

        batch=lossvec.detach().cpu().numpy()
        b_count=batch.size
        b_mean=batch.mean()
        b_M2 = ((batch - b_mean) ** 2).sum()
        delta = b_mean - mean
        total = count + b_count    
        mean += delta * b_count / total
        M2 += b_M2 + delta**2 * count * b_count / total    
        count = total
       
        print("current ste")
        print(np.sqrt(M2/(total-1))/np.sqrt(total))
        print("current mean")
        print(mean)
        gc.collect()
        torch.cuda.empty_cache()
        if j>100 and j%500==0:
                clear_output(wait=False)
      
        del ys_tray, lossvec,  obj, dif
        
    variance = M2/ (count-1)
    ste= np.sqrt(variance)/np.sqrt(count)
   
    global_means.append(mean)
    global_stds.append(ste)

    
    with torch.no_grad():
        torch.cuda.empty_cache()
    gc.collect()

/home/qtomcat/anaconda3/lib/python3.12/site-packages/hypernn/torch/utils.py:35: FutureWarning: We've integrated functorch into PyTorch. As the final step of the integration, `functorch.make_functional` is deprecated as of PyTorch 2.0 and will be deleted in a future version of PyTorch >= 2.3. Please use `torch.func.functional_call` instead; see the PyTorch 2.0 release notes and/or the `torch.func` migration guide for more details https://pytorch.org/docs/main/func.migrating.html
  _functional, named_params = make_functional(module)
/home/qtomcat/Desktop/IJCAI/Networks/utils.py:659: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be 

current ste
0.3948771007537418
current mean
81.49996185302734
current ste
0.2838884958960236
current mean
81.64085388183594


  4%|▍         | 4/100 [00:00<00:17,  5.58it/s]

current ste
0.228347339143158
current mean
81.49839274088542
current ste
0.19903080586762026
current mean
81.60611152648926


  6%|▌         | 6/100 [00:01<00:16,  5.68it/s]

...

current ste
0.051428676420981714
current mean
81.63172674960776
current ste
0.050975003471300656
current mean
81.64261454920614


Exception ignored in: <function _xla_gc_callback at 0x7f56bb1b5940>
Traceback (most recent call last):
  File "/home/qtomcat/anaconda3/lib/python3.12/site-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 
 63%|██████▎   | 63/100 [00:10<00:06,  5.88it/s]Exception ignored in: <function _xla_gc_callback at 0x7f56bb1b5940>
Traceback (most recent call last):
  File "/home/qtomcat/anaconda3/lib/python3.12/site-packages/jax/_src/lib/__init__.py", line 96, in _xla_gc_callback
    def _xla_gc_callback(*args):
    
KeyboardInterrupt: 
 64%|██████▍   | 64/100 [00:11<00:06,  5.85it/s]

current ste
0.050580169622368984
current mean
81.6299342806377
current ste
0.05020163823373575
current mean
81.63057434558867


 64%|██████▍   | 64/100 [00:11<00:06,  5.75it/s]


KeyboardInterrupt: 